### Separamos datos y preparamos para subirlo a Supabase

### Librerías

In [5]:
import os
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from supabase import create_client, Client

In [9]:

# 1. Cargar variables de entorno buscando en directorios superiores
# find_dotenv subirá desde /backend hasta la raíz para encontrar el archivo
load_dotenv(find_dotenv('.env.local'))

SUPABASE_URL = os.getenv("NEXT_PUBLIC_SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_SERVICE_ROLE_KEY")

if not SUPABASE_URL or not SUPABASE_KEY:
    raise ValueError("⚠️ Faltan las credenciales de Supabase. Revisa que el .env.local esté bien nombrado en la raíz.")

# 2. Inicializar cliente de Supabase
supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

# 3. Carga de los DataFrames
# Al estar el excel en la misma carpeta 'backend' que el notebook, 
# la ruta directa al nombre del archivo es correcta.
ruta_excel = 'datosduros.xlsx' 
df_base = pd.read_excel(ruta_excel, sheet_name='Base con continentes')
df_presis = pd.read_excel(ruta_excel, sheet_name='Presidentes Paises')

# 4. Transformación Wide to Long
años = [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
df_gastos = df_base.melt(
    id_vars=['COUNTRY', 'CONTINENTE', 'SECTOR', 'INDICATOR', 'TYPE_OF_TRANSFORMATION', 'SCALE'],
    value_vars=años,
    var_name='fiscal_year',
    value_name='amount'
)

# Limpiamos los nulos para no subir basura a la base de datos
df_gastos = df_gastos.dropna(subset=['amount'])

# 5. Preparar diccionarios para inserción
records_presis = df_presis.to_dict(orient='records')
records_gastos = df_gastos.to_dict(orient='records')

print("✅ Entorno cargado correctamente.")
print(f"📊 Listos para insertar {len(records_presis)} presidentes y {len(records_gastos)} registros de gastos.")

✅ Entorno cargado correctamente.
📊 Listos para insertar 377 presidentes y 640457 registros de gastos.
